# The coalescent

## Setup – two options

**Option A – the provided conda environment** (all packages, isolated). In a terminal / Anaconda Prompt, in the folder containing `env.yaml`:

```bash
mamba env create -f env.yaml      # or: conda env create -f env.yaml   (slower)
```

then select that environment as the kernel of this notebook (VS Code: *Select Kernel* → *Python Environments*; Jupyter: *Kernel* → *Change kernel*).

**Option B – install into whatever Python you already have** (quick; also works on Google Colab). Uncomment the line(s) you need in the next cell and run it once. `msprime` is the one package you are unlikely to have already.

In [ ]:
# Option B: uncomment what you are missing, run once, then re-comment (or delete the cell).
# %pip install -q msprime                  # coalescent simulator (pulls in tskit); pre-built wheels, ~10 s
# %pip install -q numpy matplotlib
# %pip install -q jupyter ipykernel        # only if you run this outside VS Code / Colab and have no Jupyter yet

In [ ]:
# quick check that everything needed is available
import numpy, matplotlib, msprime
print("numpy", numpy.__version__, "| matplotlib", matplotlib.__version__, "| msprime", msprime.__version__)

In `WrightFisherTutorial.ipynb`, we simulated the frequency of an allele *forward* in time. In this tutorial, we will instead simulate the genealogy of several sampled gene copies *backward* in time. This process is known as the **coalescent** and is widely used in population genetics. One of the main advantages of coalescent simulations is that they are computationally efficient: we only keep track of the few lineages ancestral to our sample, not the whole population. One drawback is that it is very difficult to incorporate selection. We will be using [`msprime`](https://tskit.dev/msprime/docs/stable/), a popular and widely used coalescent simulator.

## How to use this notebook

Same rules as in the Wright-Fisher tutorial:

1. **🔮 Predict** before you run.
2. **▶️ Run / 💻 Code** – the coding tasks give hints (which functions to use), not the solution.
3. **🔍 Look & explain** – answer the questions *before* opening the collapsible **"Reveal answer"** boxes.

### 🔍 Warm-up: the theory you will test today (haploid version)

The lecture derived everything for a **diploid** population with $2N$ gene copies. Here we simulate with `ploidy=1`, so the population consists of $N$ gene copies and **every $2N$ in the lecture becomes $N$**. Fill in the haploid versions before you continue:

| Quantity | Lecture (diploid, $2N$ copies) | This notebook (haploid, $N$ copies) |
|---|---|---|
| P(two lineages coalesce in a given generation) | $1/2N$ | ? |
| Expected coalescence time of 2 lineages, $E[X_2]$ | $2N$ | ? |
| Variance, $\mathrm{Var}(X_2)$ | $4N^2$ | ? |
| Expected waiting time while there are $k$ lineages, $E[T_k]$ | $\dfrac{4N}{k(k-1)}$ | ? |
| $E[t_\text{MRCA}]$ for $n$ lineages | $\sum_{k=2}^{n}\dfrac{4N}{k(k-1)} = 4N\left(1 - \frac{1}{n}\right)$ | ? |

Also: *why* is the coalescence time of two lineages (approximately) exponentially distributed? What is the connection to the "1 − 1/2N" argument in the lecture?

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

| Quantity | haploid, $N$ copies |
|---|---|
| P(coalesce in a generation) | $1/N$ |
| $E[X_2]$ | $N$ |
| $\mathrm{Var}(X_2)$ | $N^2$ (standard deviation $= N$, i.e. as large as the mean!) |
| $E[T_k]$ | $\dfrac{2N}{k(k-1)}$ ($\binom{k}{2}$ pairs, each coalescing at rate $1/N$) |
| $E[t_\text{MRCA}]$ | $\sum_{k=2}^{n}\dfrac{2N}{k(k-1)} = 2N\left(1 - \frac{1}{n}\right)$ |

Two lineages either pick the same parent this generation (probability $1/N$) or they don't ($1 - 1/N$), and the same happens independently every generation. The number of generations until the first "success" is therefore geometric: $P(X_2 = t) = (1 - 1/N)^{t-1}\,(1/N)$. For large $N$, $(1 - 1/N)^t \approx e^{-t/N}$, and the geometric distribution becomes an **exponential** with rate $1/N$ – mean $N$, variance $N^2$. The key property is that the *rate is constant in time* (memoryless): the chance of coalescing in the next generation does not depend on how long you have already waited, yet the *most likely* time is the very first generation.

</details>

## Drawing some trees

Let's begin by drawing some trees for a sample of 5 gene copies from a population of size 10,000. Note that each leaf and each internal node of the tree is labelled by an integer. For more information on the parameters of `sim_ancestry`, see the [documentation](https://tskit.dev/msprime/docs/stable/api.html#msprime.sim_ancestry).

🔮 *Predict:* we ask for 3 replicate trees with identical parameters. Will they be identical? Where in the tree do you expect the branches to be long, and where short?

In [ ]:
import msprime as mp

# simulate 3 replicates of tree sequence of length 1
replicates = mp.sim_ancestry(samples=5, population_size=10000, num_replicates=3, ploidy=1)
for ts in replicates:

    # obtain first (and only) tree of the tree sequence
    tree = ts.first()

    # draw tree
    print(tree.draw_text())

### 🔍 Read the trees

1. Which integers label the sampled gene copies, and which label ancestors (coalescence events)? Which node is the MRCA of the sample?
2. Node `5` is the *first* coalescence event (most recent, looking back from the present) in every tree – it is never node `8`. Why? (Think about how `msprime` must number the nodes.)
3. In your first tree, which two samples coalesced first, and which lineage was the last to join? Compare across the three trees: do you see any pattern in *which* samples pair up? Should you?
4. `draw_text` does **not** draw branches to scale. If it did, which part of the tree would you expect to be stretched the most – the tips or the region just below the root? (Use the $E[T_k]$ table from the warm-up.)

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

1. Samples are `0`–`4` (the $n$ leaves); internal nodes `5`–`8` are the $n - 1 = 4$ coalescence events; node `8` is the root = MRCA of the sample.
2. `msprime` numbers the sample nodes first (`0 … n-1`) and then the internal nodes **in order of their time**, youngest first. Node `n` (= 5 here) is thus always the first coalescence event and node `2n − 2` (= 8) always the root. We will exploit this below with `tree.time(5)`. (This is how `msprime` happens to build its node table for a single tree; it is not a guarantee of the file format. A more robust way to find the first coalescence is `min(tree.time(u) for u in tree.nodes() if not tree.is_sample(u))`.)
3. Any answer is right for your own tree – but there is **no pattern**: under the neutral coalescent every pair of lineages is equally likely to be the next to coalesce. The tree *topology* is random and exchangeable; all 5 samples are equivalent.
4. The region just below the root: the time interval during which only two lineages are left (between node `7` and node `8`). With $k=2$ lineages there is only one pair, so the expected waiting time is $N = 10{,}000$ generations, versus $2N/(5\cdot4) = 1{,}000$ generations for the first event – so the two branches entering the root are each at least ~10 000 generations long, while the branches at the tips are typically a few hundred to a thousand. The lecture slide "sampling more sequences" showed exactly this: deep, long internal branches and a bushy, short-branched tip region.

</details>

## The time to the first coalescent event

### 💻 Coding task 1

Write a function

```python
def first_coalescence_times(n_samples: int, N: int, n_reps: int) -> np.ndarray:
    ...
```

that simulates `n_reps` replicate trees for `n_samples` lineages in a population of size `N` (`ploidy=1`) and returns an array with the time of the **first** coalescence event in each replicate. Use it to estimate the mean for `n_samples = 2, 5, 10` with `N = 10000` and compare with the theoretical expectation.

🔮 *Predict first:* how does the mean time to the first coalescence change when you go from 2 to 5 to 10 samples? By what factor?

**Hints**
- `mp.sim_ancestry(...)` requires **keyword arguments**: only `samples` may be given positionally, everything else must be named (`population_size=N, num_replicates=n_reps, ploidy=1`), otherwise you get `TypeError: sim_ancestry() takes from 0 to 1 positional arguments`.
- With `num_replicates=n_reps` it returns a *generator* of tree sequences; you can iterate over it in a list comprehension: `[... for ts in mp.sim_ancestry(...)]`.
- For each tree sequence, `ts.first()` gives the (only) tree and `tree.time(u)` gives the time of node `u` ([documentation](https://tskit.dev/tskit/docs/stable/python-api.html#tskit.Tree.time)). Which node id is the first coalescence for a sample of size `n_samples`? (See question 2 above – it is *not* always 5.)
- Wrap the result in `np.array(...)` so you can call `.mean()` and `.var()` on it.
- Use at least 1000 replicates – the standard deviation of these times is as large as their mean, so the mean of few replicates is very noisy.

In [ ]:
import numpy as np

# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
def first_coalescence_times(n_samples: int, N: int, n_reps: int) -> np.ndarray:
    # the first coalescence event is always node number n_samples
    return np.array([
        ts.first().time(n_samples)
        for ts in mp.sim_ancestry(samples=n_samples, population_size=N, num_replicates=n_reps, ploidy=1)
    ])


N = 10000
for n in [2, 5, 10]:
    times = first_coalescence_times(n, N, n_reps=2000)
    expected = 2 * N / (n * (n - 1))
    print(f"n = {n:>2}: mean = {times.mean():8.0f}   expected = {expected:8.0f}   sd = {times.std():8.0f}")
```

Expected: **10 000, 1 000, 222** generations for $n = 2, 5, 10$ (the haploid $E[T_n] = 2N / (n(n-1))$). Going from 2 to 5 samples makes the first event **10× faster** (1 pair → 10 pairs), from 5 to 10 another **4.5×** (10 → 45 pairs). Your means should be within a few percent of these; note that the standard deviation is about equal to the mean in every case, as expected for an exponential distribution.

</details>

### 💻 Bonus: plot the distribution of the time to the first coalescence

🔮 *Predict first:* sketch the histogram for `n_samples = 5`. Where is its peak – at 0, at the mean (1 000), or somewhere else? Is it symmetric?

Plot a histogram of ~5000 replicates (`n_samples=5, N=10000`) and overlay the theoretical density.

**Hints**
- `plt.hist(times, bins=50, density=True)` gives a normalised histogram.
- The theory says the times are exponential with mean $\mu = 2N/(n(n-1))$; its density is $f(t) = \frac{1}{\mu} e^{-t/\mu}$. Evaluate it on `t = np.linspace(0, times.max(), 200)` and `plt.plot` it on top.

In [ ]:
import matplotlib.pyplot as plt

# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
n, N = 5, 10000
times = first_coalescence_times(n, N, n_reps=5000)
mu = 2 * N / (n * (n - 1))

plt.hist(times, bins=50, density=True, label="simulation")
t = np.linspace(0, times.max(), 200)
plt.plot(t, np.exp(-t / mu) / mu, label="exponential, mean 2N/(n(n-1))")
plt.xlabel("time to first coalescence (generations)")
plt.ylabel("density")
plt.legend();
```

The histogram is a decaying exponential: the **mode is at 0**, the mean is 1 000, the median ($\mu \ln 2 \approx 693$) is *below* the mean, and there is a long right tail (a few replicates wait > 4 000 generations). This is what "memoryless" looks like: in every generation the 10 pairs have the same chance $10/N = 0.001$ of coalescing, so the most probable single generation is always the very next one, even though on average you wait 1 000 of them. Do *not* confuse "expected time 1 000" with "it usually happens around generation 1 000".

</details>

## The height of the tree ($t_\text{MRCA}$)

### 💻 Coding task 2

Now determine the **mean and variance of the tree height** (the time of the root = $t_\text{MRCA}$) for `n_samples = 5, N = 10000`, and compare with the theoretical expectation.

Then repeat the *mean* for `n_samples = 2, 5, 10, 50`.

🔮 *Predict first:* if you go from 5 to 50 samples (10× more), does the expected $t_\text{MRCA}$ go up 10×, 2×, or barely at all? And how does it compare with what happened to the *first* coalescence time in task 1?

**Hints**
- `tree.root` is the id of the root node, so `tree.time(tree.root)` is the height. (Alternatively, use the node-numbering trick from before.)
- Write a function `tree_heights(n_samples, N, n_reps)` modelled on `first_coalescence_times`.
- The theoretical variance is the sum of the variances of the independent intervals: $\mathrm{Var}(t_\text{MRCA}) = \sum_{k=2}^{n} \left(\frac{2N}{k(k-1)}\right)^2$. Compute it with a short `sum(... for k in range(2, n + 1))`.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
def tree_height(ts) -> float:
    # height of the (only) tree in a tree sequence
    tree = ts.first()
    return tree.time(tree.root)


def tree_heights(n_samples: int, N: int, n_reps: int) -> np.ndarray:
    return np.array([
        tree_height(ts)
        for ts in mp.sim_ancestry(samples=n_samples, population_size=N, num_replicates=n_reps, ploidy=1)
    ])


N = 10000

heights = tree_heights(5, N, n_reps=2000)
exp_mean = sum(2 * N / (k * (k - 1)) for k in range(2, 6))
exp_var = sum((2 * N / (k * (k - 1))) ** 2 for k in range(2, 6))
print(f"n = 5: mean = {heights.mean():.0f} (expected {exp_mean:.0f}),  "
      f"sd = {heights.std():.0f} (expected {np.sqrt(exp_var):.0f})")

for n in [2, 5, 10, 50]:
    h = tree_heights(n, N, n_reps=1000)
    print(f"n = {n:>2}: mean tMRCA = {h.mean():7.0f}   expected = {2 * N * (1 - 1 / n):7.0f}")
```

- For $n = 5$: $E[t_\text{MRCA}] = 2N(1 - 1/5) = 16\,000$ and $\mathrm{sd} \approx 10\,700$ – again the spread is almost as large as the mean.
- For $n = 2, 5, 10, 50$ the expected heights are **10 000, 16 000, 18 000, 19 600**: going from 5 to 50 samples adds only ~20 %. $t_\text{MRCA}$ **saturates at $2N$** (haploid; $4N$ in lecture notation).
- Contrast with task 1: the *first* coalescence got 45× faster from $n = 2$ to $n = 10$, but the *root* barely moved. Adding samples only adds short branches at the tips; the deep part of the tree – where most of the time (and most of the mutations) is – is already captured with a handful of samples. This is the lecture's point that with ~5 sequences you have probably already sampled the MRCA of the whole population (the probability that a sample of $n$ shares its MRCA with the entire population is $(n-1)/(n+1)$, i.e. $2/3$ for $n = 5$).

</details>

### 💻 Coding task 3 – where is the time in the tree?

For `n_samples = 5`, estimate what **fraction of the tree height** is taken up by the *last* interval, i.e. the time during which only two lineages remain (from the second-to-last coalescence to the root).

🔮 *Predict first* from the $E[T_k]$ table: $E[T_2] / E[t_\text{MRCA}]$ = ?

**Hints**
- Node times are *ages* (generations ago), so the last interval is `tree.time(tree.root)` minus the age of the **older** of the two children of the root – the second-to-last coalescence event. `tree.children(u)` returns the children of node `u`; `max(tree.time(c) for c in tree.children(tree.root))` gives the age of the older child. (Careful: `min` would give the younger child, which may be a leaf with age 0 – then you would just get the whole height back.)
- Compute the ratio per replicate and average, or average numerator and denominator separately – do you get the same answer? (Why might they differ slightly? $E[A/B] \neq E[A]/E[B]$ in general.)

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
n, N = 5, 10000
last, height = [], []

for ts in mp.sim_ancestry(samples=n, population_size=N, num_replicates=2000, ploidy=1):
    tree = ts.first()
    root_time = tree.time(tree.root)
    older_child = max(tree.time(c) for c in tree.children(tree.root))   # second-to-last coalescence
    last.append(root_time - older_child)
    height.append(root_time)

last, height = np.array(last), np.array(height)
print("E[last interval] / E[height] =", last.mean() / height.mean())
print("mean of per-tree ratios       =", (last / height).mean())
```

Theory: $E[T_2]/E[t_\text{MRCA}] = N / (2N(1 - 1/5)) = 1/1.6 = 0.625$. So with 5 samples, **more than 60 % of the tree's height is the wait for the final two lineages to find each other** – and since mutations fall on branches in proportion to their length, a large share of the variation in a sample sits on those two deep branches. (The mean of per-tree ratios comes out lower, ≈ 0.54, because the ratio is a nonlinear function of two random quantities; both are legitimate summaries, but they answer slightly different questions – "what share of the expected height" vs. "the expected share of the height".)

</details>

## Adding population structure

We will now add some demography to our simulations, namely two subpopulations with migration between them. We start with two lineages in each subpopulation. Coalescent events can only take place *within* a subpopulation, so two lineages that sit in different subpopulations must wait until one of them migrates before they can coalesce.

In [ ]:
demography = mp.Demography()
demography.add_population(name="pop_0", initial_size=10000)
demography.add_population(name="pop_1", initial_size=10000)
demography.set_symmetric_migration_rate(populations=["pop_0", "pop_1"], rate=1e-6)

# list(...) so that we can loop over the three replicates more than once
# (sim_ancestry with num_replicates returns a one-shot generator)
replicates = list(mp.sim_ancestry(
    samples={'pop_0': 2, 'pop_1': 2},
    num_replicates=3,
    ploidy=1,
    demography=demography
))

### 🔮 Predict before drawing

Here the migration rate ($10^{-6}$ per lineage per generation) is much smaller than the coalescence rate for a pair within a subpopulation ($1/N = 10^{-4}$).

1. Which pairs of samples do you expect to coalesce first? (Samples are numbered in the order of the `samples` dictionary: `0, 1` are from `pop_0` and `2, 3` from `pop_1`.)
2. Sketch the topology you expect. Is it *always* going to be that topology?
3. Rough order of magnitude for the tree height: 10⁴, 10⁵, 10⁶ generations? (How long, on average, until *one* of two lineages migrates if each migrates at rate $10^{-6}$?)

### 💻 Coding task 4
Draw the three trees (as in the first cell) **and print the height of each** next to it, to check your predictions.

**Hints**
- Loop over `replicates` exactly as in the first cell; `tree.draw_text()` and `tree.time(tree.root)`.
- To check which population a sample belongs to, `tree.population(u)` returns the population id of node `u`, or `ts.samples(population=0)` lists the samples from population 0 ([docs](https://tskit.dev/tskit/docs/stable/python-api.html#tskit.TreeSequence.samples)).

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
for ts in replicates:
    tree = ts.first()
    print(tree.draw_text())
    print("samples in pop_0:", ts.samples(population=0),
          "  samples in pop_1:", ts.samples(population=1))
    print("tree height:", round(tree.time(tree.root)), "generations\n")
```

1. & 2. Almost always `(0, 1)` coalesce with each other and `(2, 3)` with each other, giving the topology `((0,1),(2,3))` – two clades matching the two populations. It is not *guaranteed*: with small probability a lineage migrates before the within-population pair has coalesced (probability roughly $2m / (2m + 1/N) \approx 2 \%$ per pair), and then you can get e.g. `((0,2),(1,3))`. Population structure shows up as **topology**, not just as branch lengths.
3. Within-population coalescences take ~$N = 10^4$ generations. After that, two lineages sit in different populations and can only coalesce once one of them migrates: two lineages each migrating at rate $10^{-6}$ means waiting on average $1/(2 \times 10^{-6}) = 500\,000$ generations, and then a further ~$N$ to coalesce. Tree heights of **~5 × 10⁵ generations** – about 17× the expectation of $2 \cdot 20\,000 \cdot (1 - 1/4) = 30\,000$ for 4 lineages drawn from a single well-mixed population of the same total size (20 000). Structure makes the deep branches enormously long: the two populations look like two almost separate species, and if we added mutations, samples from different populations would differ at many more sites than samples from the same population.

</details>

### 💻 Coding task 5 – how much migration makes two populations behave as one?

Estimate the **mean tree height** for migration rates `1e-6, 1e-5, 1e-4, 1e-3, 1e-2` (same demography otherwise: two populations of 10 000, 2 samples each) and plot it against the migration rate on a log x-axis.

🔮 *Predict first:* at very high migration the two subpopulations effectively form **one** population. Of what size? What is the expected $t_\text{MRCA}$ of 4 lineages in that population? Around which value of $N m$ (population size × migration rate) do you expect the transition?

**Hints**
- Write a function `mean_height_with_migration(rate, n_reps)` that builds the `Demography` (three lines above), runs `sim_ancestry` with `num_replicates=n_reps`, and returns the mean of `tree.time(tree.root)`. (If you wrote a `tree_height(ts)` helper in task 2, reuse it; the answer below assumes one exists.)
- Loop over the rates, collect the means, `plt.plot(rates, means, 'o-')`, `plt.xscale('log')`.
- Also draw a horizontal reference line (`plt.axhline`) at the panmictic expectation.
- 200 replicates per rate is enough for a first look.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
def mean_height_with_migration(rate: float, n_reps: int, N: int = 10000) -> float:
    demography = mp.Demography()
    demography.add_population(name="pop_0", initial_size=N)
    demography.add_population(name="pop_1", initial_size=N)
    demography.set_symmetric_migration_rate(populations=["pop_0", "pop_1"], rate=rate)
    heights = [
        tree_height(ts)
        for ts in mp.sim_ancestry(samples={'pop_0': 2, 'pop_1': 2}, num_replicates=n_reps,
                                  ploidy=1, demography=demography)
    ]
    return np.mean(heights)


rates = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
means = [mean_height_with_migration(r, n_reps=200) for r in rates]

plt.plot(rates, means, 'o-', label="two populations with migration")
plt.axhline(2 * 20000 * (1 - 1 / 4), linestyle='--', color='grey', label="one population of 20 000")
plt.xscale('log')
plt.yscale('log')
plt.xlabel("migration rate m (per lineage per generation)")
plt.ylabel("mean tree height (generations)")
plt.legend();
```

At high migration the sample behaves as if drawn from a single panmictic population of size $2 \times 10\,000 = 20\,000$ (two demes of $N$ each – not to be confused with the diploid $2N$ of the lecture), whose expected height for 4 lineages is $2 \cdot 20\,000 \cdot (1 - 1/4) = 30\,000$. Typical values: ≈ 500 000, 65 000, 35 000, 30 000, 30 000 generations for $m = 10^{-6} \ldots 10^{-2}$. The transition happens around $N m \approx 0.1$–$1$ (i.e. $m \approx 10^{-5}$–$10^{-4}$ here): once there is roughly **one migrant per generation** ($Nm = 1$; the lecture's diploid "$2Nm$" is the same idea) migration is as fast as coalescence, and the tree is already within ~15 % of the panmictic value. Below that ($Nm \ll 1$) the height is dominated by the wait for a migration event, $\approx 1/(2m)$, and grows as $m$ shrinks. This is why "one migrant per generation" is a famous rule of thumb for when subpopulations stop diverging by drift.

</details>

## Bonus: a bottleneck

The lecture discussed bottlenecks and founder effects (elephant seals, black spruce) and the idea of an *effective* population size. Let us see what a bottleneck does to genealogies.

### 💻 Coding task 6

Simulate a single population of size 10 000 that, going **backwards** in time, was of size **100 between 1 000 and 1 500 generations ago** (and 10 000 before that). Sample 5 lineages, collect ~2000 tree heights and plot their histogram. Compare with the histogram for a constant population of 10 000.

🔮 *Predict first:*
- How many of the 5 lineages do you expect to have coalesced *before* (more recently than) 1 000 generations ago? (Use $E[t_\text{MRCA}]$ and $E[T_5]$ for $N = 10\,000$.)
- Once the remaining lineages enter the bottleneck, how quickly do they coalesce? ($E[T_k]$ with $N = 100$.) Will any lineages "survive" the 500 generations of the bottleneck?
- So what shape will the histogram have? Where is its peak?

**Hints**
- Population size changes are added with `demography.add_population_parameters_change(time=..., initial_size=..., population="pop_0")` ([docs](https://tskit.dev/msprime/docs/stable/api.html#msprime.Demography.add_population_parameters_change)). `time` is in generations *ago*. You need two such changes: one at 1 000 (size becomes 100) and one at 1 500 (size becomes 10 000 again).
- Then `mp.sim_ancestry(samples=5, demography=demography, num_replicates=..., ploidy=1)`; note that you pass `demography`, not `population_size`. Reuse your `tree_heights` / `tree_height` helpers from task 2 for collecting heights.
- `plt.hist([heights_constant, heights_bottleneck], bins=60, label=[...])`; a log-scaled x-axis (`plt.xscale('log')`) makes the two very different scales visible at once.
- Also print the mean height in both cases.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
demography = mp.Demography()
demography.add_population(name="pop_0", initial_size=10000)
demography.add_population_parameters_change(time=1000, initial_size=100, population="pop_0")
demography.add_population_parameters_change(time=1500, initial_size=10000, population="pop_0")

heights_bottleneck = np.array([
    tree_height(ts)
    for ts in mp.sim_ancestry(samples=5, demography=demography, num_replicates=2000, ploidy=1)
])
heights_constant = tree_heights(5, 10000, n_reps=2000)

print("mean height, constant N = 10000:", heights_constant.mean())
print("mean height, bottleneck:        ", heights_bottleneck.mean())

bins = np.logspace(2.5, 5.5, 60)   # 300 ... 300 000 generations, equal width on a log axis
plt.hist([heights_constant, heights_bottleneck], bins=bins, label=["constant N = 10 000", "bottleneck"])
plt.xscale('log')
plt.xlabel("tree height (generations)")
plt.ylabel("number of replicates")
plt.legend();
```

- Before the bottleneck (the most recent 1 000 generations) the 5 lineages behave as in a population of 10 000: the first event takes ~1 000 generations on average, so typically **0–2** coalescences have happened, and 3–5 lineages enter the bottleneck.
- Inside the bottleneck, with $N = 100$, $E[T_5] = 10$, $E[T_3] = 33$ and $E[T_2] = 100$ generations: the expected time to the MRCA of 5 lineages is only 160 generations, so within the 500-generation bottleneck essentially **every replicate reaches its MRCA** (only ≈ 1–2 % of replicates still have two lineages left after 500 generations: $P(T_2 > 400) = e^{-4} \approx 0.02$). Very few lineages survive into the pre-bottleneck period.
- Hence the histogram of the bottleneck population is a **sharp peak just after 1 000 generations** (about 90 % of heights between 1 000 and 1 300), compared with the broad distribution (mean 16 000, ranging from < 2 000 to > 40 000) for the constant population. The mean drops from 16 000 to roughly 1 200–1 300 (the few replicates that escape the bottleneck pull the mean up a bit).

**Connection to the lecture.** A short, sharp bottleneck wipes out the deep part of the genealogy: all lineages are forced through the few individuals alive during the bottleneck, so almost all pre-bottleneck variation is lost (the elephant seals with only two mtDNA alleles). If you only had the tree heights, you would infer a population that is *much* smaller than the 10 000 it is today – its **effective** size $N_e$ is dragged down by the bottleneck. For sizes that fluctuate quickly, $N_e$ is the harmonic mean of the sizes over time, which is dominated by the smallest values: a few hundred generations at $N = 100$ outweigh many thousands at $N = 10\,000$.

</details>

## Wrap-up: connect the simulations to the lecture

1. Forward-in-time Wright-Fisher simulations and backward-in-time coalescent simulations are two views of the same model. In the WF tutorial, drift fixed or lost an allele in a time proportional to $N$; in this notebook, lineages coalesced in a time proportional to $N$. Explain in one sentence why those are the same statement.
2. Give two reasons why the coalescent is so much cheaper to simulate than the forward Wright-Fisher model.
3. Why is it hard to include selection in a coalescent simulation, given how the process was constructed (all lineages exchangeable, coalescence rate depends only on $k$ and $N$)?
4. You sequence 5 individuals from a population and find that the genealogy has one extremely long internal branch separating two groups of samples. Name two different demographic scenarios from this notebook that could produce this, and how you might distinguish them with more samples.

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

1. An allele is fixed exactly when *all* gene copies in the population descend from a single ancestral copy – i.e. when the lineages of the whole population have coalesced. The forward "time to fixation" and the backward "time to the MRCA" are the same waiting time seen from the two ends.
2. (i) You only track the $n$ lineages ancestral to the *sample*, not all $N$ individuals (and $n$ shrinks to 1 as you go back); (ii) you jump straight from one coalescence event to the next by drawing an exponential waiting time, instead of simulating every one of the ~$N$ intervening generations.
3. Under selection lineages are *not* exchangeable: a lineage carrying the favoured allele has more offspring, so its ancestry looks different, and the coalescence rate depends on the allele frequency trajectory in the whole population – which the coalescent, by design, does not track. (Special tricks exist, but they lose most of the simplicity.)
4. Population structure with little migration (two subpopulations, samples from both) or a plain neutral coalescent in one population – remember that even without structure, $T_2$ alone is > 60 % of the tree height, so a long deep branch is *expected*. With more samples: under structure the two clades keep corresponding to the two locations and stay separated by the long branch; in a single panmictic population, added samples join both sides at random and the topology carries no geographic signal.

</details>